# Injini fault-ID-only rerun — streamed embedding, progress printed

In [ ]:

import os, sys, glob, subprocess
WORK = "/tmp/injini"
for d in ("src", "models", "data"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, os.path.join(WORK, "src"))

In [ ]:
%%writefile src/features.py
"""Log-mel front end for Injini.

The embedder Injini ships is an EfficientAT MobileNetV3 pretrained on AudioSet.
Those weights are only valid against EfficientAT's own mel spec, so this module
reproduces that spec exactly:

    sample rate 32 kHz, pre-emphasis 0.97, STFT n_fft=1024 / hop=320 /
    win_length=800 (symmetric Hann, zero-padded to n_fft, centered/reflect),
    128 Kaldi-style mel bands 0-15000 Hz, log(mel + 1e-5), then (x + 4.5) / 5.

Two implementations of that one spec:

  * ``reference_logmel`` uses EfficientAT's ``AugmentMelSTFT`` in eval mode. This
    is the ground truth used for training and evaluation on Kaggle.
  * ``logmel`` is a pure-NumPy reimplementation with no torch / torchaudio at
    run time. It loads a precomputed Kaldi mel matrix (``models/mel_kaldi_128x513.npy``,
    written by ``dump_mel_matrix``) so the fiddly Kaldi filterbank never has to
    be re-derived. This is the version ported to Kotlin for the phone, and
    ``tests/test_feature_parity.py`` holds the two within 1e-3.

Capture on the phone is 16 kHz (matches SiloSense, matches the Android
UNPROCESSED source, keeps knock detection honestly out of scope). ``load_audio``
resamples whatever it is given to 32 kHz with the same naive linear interp
SiloSense used.
"""
from __future__ import annotations

import os

import numpy as np
import soundfile as sf

SR = 32_000
N_FFT = 1024
HOP = 320
WIN_LENGTH = 800
N_MELS = 128
FMIN = 0.0
FMAX = 15_000.0  # AugmentMelSTFT eval: sr // 2 - fmax_aug_range // 2 = 16000 - 1000
PREEMPH = 0.97
LOG_OFFSET = 1e-5
NORM_ADD = 4.5
NORM_DIV = 5.0

CLIP_SECONDS = 10.0
CLIP_SAMPLES = int(SR * CLIP_SECONDS)
N_FRAMES = 1 + CLIP_SAMPLES // HOP  # 1001, the fixed length used for the on-device ONNX graph

_HERE = os.path.dirname(os.path.abspath(__file__))
MEL_MATRIX_PATH = os.path.join(_HERE, os.pardir, "models", "mel_kaldi_128x513.npy")

# Symmetric (periodic=False) Hann of win_length, zero-padded to n_fft and centered,
# matching torch.stft(win_length=800, n_fft=1024, window=hann_window(800, periodic=False)).
_hann = 0.5 - 0.5 * np.cos(2.0 * np.pi * np.arange(WIN_LENGTH) / (WIN_LENGTH - 1))
_WINDOW = np.zeros(N_FFT, dtype=np.float64)
_pad_left = (N_FFT - WIN_LENGTH) // 2
_WINDOW[_pad_left:_pad_left + WIN_LENGTH] = _hann

_MEL: np.ndarray | None = None


def dump_mel_matrix(path: str = MEL_MATRIX_PATH) -> np.ndarray:
    """Compute EfficientAT's Kaldi mel filterbank once (needs torchaudio) and cache it.

    Returns a (128, 513) float32 matrix: get_mel_banks(...) padded with one zero
    column, exactly as AugmentMelSTFT does before ``mel_basis @ power``.
    """
    import torch
    import torchaudio

    mel_basis, _ = torchaudio.compliance.kaldi.get_mel_banks(
        N_MELS, N_FFT, SR, FMIN, FMAX,
        vtln_low=100.0, vtln_high=-500.0, vtln_warp_factor=1.0,
    )
    mel_basis = torch.nn.functional.pad(mel_basis, (0, 1), mode="constant", value=0)
    mat = mel_basis.cpu().numpy().astype(np.float32)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.save(path, mat)
    return mat


def mel_matrix() -> np.ndarray:
    global _MEL
    if _MEL is None:
        if not os.path.exists(MEL_MATRIX_PATH):
            dump_mel_matrix(MEL_MATRIX_PATH)
        _MEL = np.load(MEL_MATRIX_PATH).astype(np.float64)
    return _MEL


def resample_linear(y: np.ndarray, orig_sr: int, target_sr: int = SR) -> np.ndarray:
    if orig_sr == target_sr or len(y) == 0:
        return y.astype(np.float32)
    duration = len(y) / orig_sr
    n_target = max(1, int(round(duration * target_sr)))
    x_orig = np.linspace(0.0, duration, num=len(y), endpoint=False)
    x_target = np.linspace(0.0, duration, num=n_target, endpoint=False)
    return np.interp(x_target, x_orig, y).astype(np.float32)


def load_audio(path: str, offset: float = 0.0, duration: float | None = None) -> np.ndarray:
    with sf.SoundFile(path) as f:
        sr = f.samplerate
        f.seek(int(offset * sr))
        n = int(duration * sr) if duration is not None else -1
        y = f.read(frames=n, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    return resample_linear(np.asarray(y, dtype=np.float32), sr, SR)


def fixed_length(y: np.ndarray, n: int = CLIP_SAMPLES) -> np.ndarray:
    if len(y) >= n:
        return y[:n]
    return np.pad(y, (0, n - len(y)), mode="constant")


def _preemphasis(y: np.ndarray) -> np.ndarray:
    # torch: conv1d(x, [[[-.97, 1]]]) -> out[t] = -0.97*x[t] + x[t+1], length N-1.
    return (y[1:] - PREEMPH * y[:-1]).astype(np.float64)


def _stft_power(y: np.ndarray) -> np.ndarray:
    pad = N_FFT // 2
    yp = np.pad(y, (pad, pad), mode="reflect")
    n_frames = 1 + (len(yp) - N_FFT) // HOP
    idx = np.arange(N_FFT)[:, None] + HOP * np.arange(n_frames)[None, :]
    frames = yp[idx] * _WINDOW[:, None]
    spec = np.fft.rfft(frames, n=N_FFT, axis=0)
    return (spec.real ** 2 + spec.imag ** 2)  # (513, n_frames)


def logmel(y: np.ndarray, fixed: bool = False) -> np.ndarray:
    """Pure-NumPy log-mel matching ``reference_logmel``. (128, T) float32.

    ``fixed=True`` pins the output to ``N_FRAMES`` columns for the on-device graph.
    """
    if fixed:
        y = fixed_length(y)
    power = _stft_power(_preemphasis(np.asarray(y, dtype=np.float64)))
    mel = mel_matrix() @ power
    mel = np.log(mel + LOG_OFFSET)
    mel = (mel + NORM_ADD) / NORM_DIV
    out = mel.astype(np.float32)
    if fixed:
        if out.shape[1] >= N_FRAMES:
            out = out[:, :N_FRAMES]
        else:
            out = np.pad(out, ((0, 0), (0, N_FRAMES - out.shape[1])), mode="edge")
    return out


def reference_logmel(y: np.ndarray) -> np.ndarray:
    """EfficientAT AugmentMelSTFT in eval mode. Ground truth. Needs torch/torchaudio."""
    import torch
    import sys

    vendor = os.path.join(_HERE, os.pardir, "vendor_efficientat")
    if vendor not in sys.path:
        sys.path.insert(0, vendor)
    from models.preprocess import AugmentMelSTFT

    mel = AugmentMelSTFT(
        n_mels=N_MELS, sr=SR, win_length=WIN_LENGTH, hopsize=HOP, n_fft=N_FFT,
        freqm=0, timem=0, fmin=FMIN, fmax=FMAX, fmin_aug_range=1, fmax_aug_range=1,
    )
    mel.eval()
    with torch.no_grad():
        spec = mel(torch.from_numpy(np.asarray(y, dtype=np.float32))[None, :])
    return spec.squeeze(0).cpu().numpy()


if __name__ == "__main__":
    mat = dump_mel_matrix()
    print(f"wrote {MEL_MATRIX_PATH}  shape={mat.shape}  sum={mat.sum():.3f}")

In [ ]:
%%writefile src/embedder.py
"""Frozen EfficientAT MobileNetV3 embedder.

The state-of-the-art DCASE Task 2 recipe scores a clip by its distance, in the
embedding space of a large frozen audio network, from the healthy recordings of
that machine. Injini keeps the recipe and swaps the network for a MobileNetV3
distilled from a transformer teacher on AudioSet (EfficientAT ``mn10_as`` /
``mn04_as``), which is small enough to quantise onto an Arm phone.

This module wraps EfficientAT's ``MN`` so ``forward`` returns only the
L2-normalised embedding (``F.adaptive_avg_pool2d`` of the last feature map), and
exports that subgraph to ONNX. The anomaly score is computed outside the graph
(see ``anomaly.py``); it is a few lines of linear algebra and never needs Arm
acceleration.
"""
from __future__ import annotations

import contextlib
import io
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

_HERE = os.path.dirname(os.path.abspath(__file__))
VENDOR = os.path.join(_HERE, os.pardir, "vendor_efficientat")

WIDTH = {"mn10_as": 1.0, "mn04_as": 0.4, "mn05_as": 0.5, "mn01_as": 0.1}
EMBED_DIM = {"mn10_as": 960, "mn04_as": 384, "mn05_as": 480, "mn01_as": 96}


def _load_mn(name: str) -> nn.Module:
    # vendor_efficientat/helpers/utils.py reads metadata/class_labels_indices.csv
    # with a cwd-relative path at import time, so run the import from VENDOR.
    if VENDOR not in sys.path:
        sys.path.insert(0, VENDOR)
    cwd = os.getcwd()
    try:
        os.chdir(VENDOR)
        from models.mn.model import get_model

        with contextlib.redirect_stdout(io.StringIO()):
            model = get_model(pretrained_name=name, width_mult=WIDTH[name], head_type="mlp")
    finally:
        os.chdir(cwd)
    return model.eval()


class Embedder(nn.Module):
    """(B, 1, 128, T) log-mel  ->  (B, D) L2-normalised embedding."""

    def __init__(self, name: str = "mn10_as", l2: bool = True):
        super().__init__()
        self.name = name
        self.l2 = l2
        self.features = _load_mn(name).features
        for p in self.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1)).flatten(1)
        if self.l2:
            x = F.normalize(x, dim=1)
        return x


def export_onnx(name: str, out_path: str, opset: int = 17) -> str:
    from features import N_FRAMES, N_MELS

    model = Embedder(name)
    dummy = torch.randn(1, 1, N_MELS, N_FRAMES)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    torch.onnx.export(
        model, dummy, out_path,
        input_names=["logmel"], output_names=["embedding"],
        dynamic_axes={"logmel": {0: "batch", 3: "frames"}, "embedding": {0: "batch"}},
        opset_version=opset, dynamo=False,
    )
    return out_path


def param_counts(name: str) -> dict:
    m = Embedder(name)
    total = sum(p.numel() for p in m.parameters())
    return {"embedder": name, "embed_dim": EMBED_DIM[name], "params": total}


if __name__ == "__main__":
    import argparse

    sys.path.insert(0, _HERE)
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="mn10_as", choices=list(WIDTH))
    ap.add_argument("--out", default=None)
    args = ap.parse_args()
    out = args.out or os.path.join(_HERE, os.pardir, "models", f"injini_{args.name}_fp32.onnx")
    export_onnx(args.name, out)
    size = os.path.getsize(out) / 1e6
    print({**param_counts(args.name), "onnx_mb": round(size, 3), "path": out})

In [ ]:
%%writefile src/embed_backends.py
"""Uniform embedding interface over the three backends the eval compares.

  * ``torch:<name>``   frozen EfficientAT MobileNetV3 in PyTorch (mn10_as / mn04_as)
  * ``onnx:<path>``    an exported / quantised embedder run through onnxruntime
  * ``passt``          the transformer teacher, the full-size reference ceiling
                       (needs ``hear21passt``; only used on Kaggle)

All backends take a list of waveforms at ``features.SR`` and return an
(N, D) float32 array of L2-normalised embeddings.
"""
from __future__ import annotations

import os
import sys

import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)

import features as F  # noqa: E402


def _mel_batch(waves: list[np.ndarray]) -> np.ndarray:
    mels = [F.logmel(w, fixed=True) for w in waves]
    return np.stack(mels, axis=0)[:, None, :, :].astype(np.float32)  # (n,1,128,T)


# Chunk size for large corpora (e.g. the ~20k-clip engine-sounds manifest):
# materialising every clip's log-mel before batching for inference held
# ~9 GB per 17k-clip split and was OOM-killed with no traceback on Kaggle's
# CPU tier. Embedding is now genuinely streamed, one chunk of mels at a time.
CHUNK = 512


class TorchBackend:
    def __init__(self, name: str):
        import torch
        from embedder import Embedder

        self.torch = torch
        self.model = Embedder(name)

    def embed(self, waves: list[np.ndarray], batch: int = 16) -> np.ndarray:
        out = []
        for c in range(0, len(waves), CHUNK):
            x = _mel_batch(waves[c:c + CHUNK])
            for i in range(0, len(x), batch):
                t = self.torch.from_numpy(x[i:i + batch])
                with self.torch.no_grad():
                    out.append(self.model(t).cpu().numpy())
        return np.concatenate(out, axis=0).astype(np.float32)


class OnnxBackend:
    def __init__(self, path: str, providers: list[str] | None = None):
        import onnxruntime as ort

        providers = providers or ["CPUExecutionProvider"]
        self.sess = ort.InferenceSession(path, providers=providers)
        self.iname = self.sess.get_inputs()[0].name

    def embed(self, waves: list[np.ndarray], batch: int = 16) -> np.ndarray:
        out = []
        for c in range(0, len(waves), CHUNK):
            x = _mel_batch(waves[c:c + CHUNK])
            for i in range(0, len(x), batch):
                out.append(self.sess.run(None, {self.iname: x[i:i + batch]})[0])
        e = np.concatenate(out, axis=0).astype(np.float32)
        return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-12)


class PasstBackend:
    def __init__(self):
        import torch
        from hear21passt.base import load_model, get_scene_embeddings

        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        try:
            _ = torch.zeros(1, device=self.device)  # trip sm_60 mismatch early
        except Exception:
            self.device = "cpu"
        self.model = load_model(mode="embed_only").eval().to(self.device)
        self._embed = get_scene_embeddings

    def embed(self, waves: list[np.ndarray], batch: int = 8) -> np.ndarray:
        out = []
        for i in range(0, len(waves), batch):
            chunk = waves[i:i + batch]
            n = max(len(w) for w in chunk)
            arr = np.zeros((len(chunk), n), dtype=np.float32)
            for j, w in enumerate(chunk):
                arr[j, : len(w)] = w
            t = self.torch.from_numpy(arr).to(self.device)
            with self.torch.no_grad():
                out.append(self._embed(t, self.model).cpu().numpy())
        e = np.concatenate(out, axis=0).astype(np.float32)
        return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-12)


def get_backend(spec: str):
    if spec == "passt":
        return PasstBackend()
    if spec.startswith("onnx:"):
        return OnnxBackend(spec.split(":", 1)[1])
    if spec.startswith("torch:"):
        return TorchBackend(spec.split(":", 1)[1])
    raise ValueError(f"unknown backend spec: {spec!r}")

In [ ]:
%%writefile src/prepare_engine_sounds.py
"""Turn the Kaggle 'zeyadzsm/engine-sounds' dump into a source-split manifest.

Two problems with that dataset, both handled here:

1. It ships augmented variants next to their source clips, with names like
   ``1_augmented_10_Alternator Bearing Noise.wav``, ``1_segment_0_augmented.wav``,
   ``002_Engine-A_augmented_1.wav``. Splitting those at file level leaks
   near-identical audio across train and test. Every file is reduced to a
   ``source key`` (strip augmentation / segment / trailing-index suffixes) and
   the split is done on the (class, source key) pair.
2. Two near-duplicate top folders (``Data/Data_Fixed`` and
   ``Data_AA/Data_Fixed``). Both are read; the source key is taken relative to
   the class folder so a clip appearing in both maps to one key.

Output: data/engine_sounds_manifest.csv with columns
    path, label, class_name, source_key, split
"""
from __future__ import annotations

import argparse
import csv
import glob
import os
import random
import re

AUG_PATTERNS = [
    r"_augmented(_\d+)?(_[A-Za-z].*)?$",
    r"_segment_\d+(_augmented.*)?$",
    r"_aug(_\d+)?$",
]
TRAIL_IDX = re.compile(r"(_\d+)+$")


def source_key(stem: str) -> str:
    s = stem
    for pat in AUG_PATTERNS:
        s = re.sub(pat, "", s, flags=re.IGNORECASE)
    s = TRAIL_IDX.sub("", s)
    return s.strip() or stem


def collect(root: str) -> list[dict]:
    rows = []
    classes = set()
    for wav in glob.glob(os.path.join(root, "**", "*.wav"), recursive=True):
        rel = os.path.relpath(wav, root).replace("\\", "/")
        parts = rel.split("/")
        # .../<something>/Data_Fixed/<class>/<maybe Augmented>/<file>.wav
        try:
            di = parts.index("Data_Fixed")
            cls = parts[di + 1]
        except (ValueError, IndexError):
            cls = parts[-2]
        classes.add(cls)
        stem = os.path.splitext(parts[-1])[0]
        rows.append({"path": wav, "class_name": cls, "source_key": f"{cls}::{source_key(stem)}"})
    labels = {c: i for i, c in enumerate(sorted(classes))}
    for r in rows:
        r["label"] = labels[r["class_name"]]
    return rows


def split(rows: list[dict], val_frac=0.2, seed=42) -> list[dict]:
    rng = random.Random(seed)
    keys_by_class: dict[str, list[str]] = {}
    for r in rows:
        keys_by_class.setdefault(r["class_name"], set()).add(r["source_key"])
    val_keys: set[str] = set()
    for cls, keys in keys_by_class.items():
        keys = sorted(keys)
        rng.shuffle(keys)
        n_val = max(1, round(len(keys) * val_frac))
        val_keys.update(keys[:n_val])
    for r in rows:
        r["split"] = "val" if r["source_key"] in val_keys else "train"
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, help="folder containing Data/ and Data_AA/")
    ap.add_argument("--out", default="data/engine_sounds_manifest.csv")
    args = ap.parse_args()

    rows = split(collect(args.root))
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    with open(args.out, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["path", "label", "class_name", "source_key", "split"])
        w.writeheader()
        w.writerows(rows)

    n_tr = sum(r["split"] == "train" for r in rows)
    n_va = sum(r["split"] == "val" for r in rows)
    n_keys = len({r["source_key"] for r in rows})
    print(f"{len(rows)} files  {n_keys} source keys  ->  train {n_tr} / val {n_va}")
    by_cls: dict[str, int] = {}
    for r in rows:
        by_cls[r["class_name"]] = by_cls.get(r["class_name"], 0) + 1
    for c, n in sorted(by_cls.items()):
        print(f"  {n:5d}  {c}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/faultid.py
"""Supervised fault-identity head on frozen embeddings.

Secondary mode. Where a labelled corpus covers a fault family, a small MLP on
the same frozen embedding names the likely fault. Trained and evaluated with a
source-disjoint split (see prepare_engine_sounds.py) so the macro-F1 reflects
transfer to unseen recordings, not memorised ones.

    python src/faultid.py --manifest data/engine_sounds_manifest.csv --backend torch:mn10_as
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import sys

import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)

import features as F                        # noqa: E402
from embed_backends import get_backend      # noqa: E402


def read_manifest(path: str):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    classes = sorted({r["class_name"] for r in rows})
    return rows, {c: i for i, c in enumerate(classes)}


def embed_split(backend, rows, split, cap=None):
    sel = [r for r in rows if r["split"] == split]
    if cap:
        sel = sel[:cap]
    print(f"  embedding {split}: {len(sel)} clips", flush=True)
    waves = [F.load_audio(r["path"]) for r in sel]
    emb = backend.embed(waves)
    y = np.array([int(r["label"]) for r in sel])
    print(f"  embedded {split}: {emb.shape}", flush=True)
    return emb, y


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--backend", default="torch:mn10_as")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--cap", type=int, default=None)
    args = ap.parse_args()

    import torch
    import torch.nn as nn
    from sklearn.metrics import f1_score, classification_report

    rows, cls_map = read_manifest(args.manifest)
    backend = get_backend(args.backend)

    Xtr, ytr = embed_split(backend, rows, "train", args.cap)
    Xva, yva = embed_split(backend, rows, "val", args.cap)
    n_cls = len(cls_map)

    clf = nn.Sequential(nn.Linear(Xtr.shape[1], 256), nn.ReLU(), nn.Dropout(0.3),
                        nn.Linear(256, n_cls))
    opt = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()
    Xt, yt = torch.from_numpy(Xtr).float(), torch.from_numpy(ytr).long()
    for ep in range(args.epochs):
        clf.train()
        opt.zero_grad()
        loss = lossf(clf(Xt), yt)
        loss.backward()
        opt.step()

    clf.eval()
    with torch.no_grad():
        pred = clf(torch.from_numpy(Xva).float()).argmax(1).numpy()
    macro = f1_score(yva, pred, average="macro")
    print(classification_report(yva, pred, target_names=list(cls_map), zero_division=0))

    res = {"backend": args.backend, "macro_f1": float(macro),
           "n_train": int(len(ytr)), "n_val": int(len(yva)), "classes": list(cls_map)}
    out = os.path.join(_HERE, os.pardir, "models",
                       f"faultid_{args.backend.replace(':', '_').replace('/', '_')}.json")
    json.dump(res, open(out, "w"), indent=2)
    print(json.dumps(res, indent=2))
    print("wrote", out)


if __name__ == "__main__":
    try:
        main()
    except Exception:
        import traceback
        traceback.print_exc()
        sys.exit(1)

In [ ]:

!pip -q install onnx onnxruntime 2>/dev/null
if not os.path.isdir("vendor_efficientat"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/fschmid56/EfficientAT.git", "vendor_efficientat"], check=True)
os.makedirs("vendor_efficientat/resources", exist_ok=True)
p = "vendor_efficientat/resources/mn10_as_mAP_471.pt"
if not os.path.exists(p):
    subprocess.run(["wget", "-q", "-O", p,
                    "https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/mn10_as_mAP_471.pt"], check=True)
import features as F
F.dump_mel_matrix()

## Find the real mount, export the embedder, run fault-ID with progress

In [ ]:

candidates = [d for d in glob.glob("/kaggle/input/*") if len(glob.glob(d + "/**/*.wav", recursive=True)) > 100]
print("candidate roots:", candidates)
assert candidates, "no wav-bearing input directory found"
ES = candidates[0]
!python src/embedder.py --name mn10_as --out models/injini_mn10_as_fp32.onnx
!python src/prepare_engine_sounds.py --root {ES} --out data/engine_sounds_manifest.csv
!python src/faultid.py --manifest data/engine_sounds_manifest.csv     --backend onnx:models/injini_mn10_as_fp32.onnx --epochs 80

In [ ]:

import shutil, glob as g
for f in g.glob("models/faultid_*.json"):
    shutil.copy(f, "/kaggle/working/")
print(sorted(os.listdir("/kaggle/working")))